# 05. Entrenamiento de Modelos Baseline — IBM TabFormer
**Unidad 1:** Detección de Fraude en Transacciones con Tarjeta de Crédito  
**Dataset:** IBM TabFormer (`train_split.parquet`)  
**Entorno:** Python 3.12 | scikit-learn | Joblib

---

### Objetivos del Notebook:
1. Cargar las particiones temporales procesadas (`train_split.parquet` y `test_split.parquet`).
2. Cargar el pipeline de preprocesamiento ajustado (`preprocessor.joblib`).
3. Transformar las características de entrenamiento exclusivamente con el preprocesador ajustado.
4. Entrenar el **Modelo Baseline Principal**:
   - **Regresión Logística:** con ponderación de balanceo de clases (`class_weight='balanced'`) para abordar el severo desbalance de 0.12%.
5. Entrenar el **Modelo Comparativo Académico**:
   - **Regresión Lineal:** por mínimos cuadrados ordinarios para contrastar formalmente por qué una regresión continua no es el método adecuado para clasificación binaria probabilística.
6. Registrar rigurosamente:
   - Semilla de reproducibilidad (`RANDOM_STATE = 42`).
   - Hiperparámetros exactos de cada modelo.
   - Variables predictoras utilizadas.
   - Tiempos de ajuste.
7. Serializar los modelos entrenados en `artifacts/baseline_logistic_regression.joblib` y `artifacts/linear_regression_comparison.joblib`.


In [1]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR
while not (PROJECT_ROOT / "notebooks").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from notebooks._comun.config import (
    TRAIN_DATA_FILE,
    TEST_DATA_FILE,
    PREPROCESSOR_ARTIFACT,
    LOGISTIC_MODEL_ARTIFACT,
    LINEAR_MODEL_ARTIFACT,
    RANDOM_STATE,
    SELECTED_FEATURES,
    LOGISTIC_REGRESSION_PARAMS,
    LINEAR_REGRESSION_PARAMS,
)
from notebooks._comun.data_utils import load_artifact, save_artifact
print(f"Raíz del proyecto: {PROJECT_ROOT}")


Raíz del proyecto: /home/glender/ProyectosUNi/iaupeu


In [2]:
import polars as pl
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression, LinearRegression
import sklearn

print(f"Versión de scikit-learn: {sklearn.__version__}")
print(f"Semilla de reproducibilidad fijada: {RANDOM_STATE}")

# Carga de particiones
train_df = pl.read_parquet(TRAIN_DATA_FILE)
test_df = pl.read_parquet(TEST_DATA_FILE)

print(f"Train Set: {train_df.shape[0]:,} registros | Fraudes: {train_df['is_fraud'].sum():,} ({train_df['is_fraud'].mean()*100:.3f}%)")
print(f"Test Set:  {test_df.shape[0]:,} registros | Fraudes: {test_df['is_fraud'].sum():,} ({test_df['is_fraud'].mean()*100:.3f}%)")


Versión de scikit-learn: 1.9.1
Semilla de reproducibilidad fijada: 42
Train Set: 336,387 registros | Fraudes: 2,670 (0.794%)
Test Set:  168,316 registros | Fraudes: 2,033 (1.208%)


## 1. Carga del Preprocesador y Transformación de X_train
Utilizamos el preprocesador serializado previamente (ajustado en `transformacion.ipynb`) para transformar `X_train`.


In [3]:
preprocessor = load_artifact(PREPROCESSOR_ARTIFACT)
print("Preprocesador cargado exitosamente.")

X_train_df = train_df.select(SELECTED_FEATURES).to_pandas()
y_train = train_df["is_fraud"].to_numpy()

# Transformación de características
X_train_trans = preprocessor.transform(X_train_df)
print(f"Matriz X_train transformada: {X_train_trans.shape[0]:,} filas x {X_train_trans.shape[1]} columnas")


Preprocesador cargado exitosamente.


Matriz X_train transformada: 336,387 filas x 21 columnas


## 2. Entrenamiento del Baseline Principal: Regresión Logística
Configuramos la Regresión Logística con `class_weight='balanced'`. Esto asigna pesos inversamente proporcionales a las frecuencias de clase en la función de costo:
$$w_j = \frac{N}{2 \cdot n_j}$$
evitando que el modelo colapse trivialmente prediciendo la clase mayoritaria.


In [4]:
print("Hiperparámetros de Regresión Logística:")
print(LOGISTIC_REGRESSION_PARAMS)

log_clf = LogisticRegression(**LOGISTIC_REGRESSION_PARAMS)

t0 = time.time()
log_clf.fit(X_train_trans, y_train)
t_train_log = time.time() - t0

print(f"Entrenamiento completado en {t_train_log:.2f} segundos.")
print(f"Intercepto (bias): {log_clf.intercept_[0]:.4f}")
print(f"Número de coeficientes ajustados: {len(log_clf.coef_[0])}")

# Coeficientes más significativos
feature_names = preprocessor.get_feature_names_out()
coef_df = pd.DataFrame({
    "Característica": feature_names,
    "Coeficiente": log_clf.coef_[0],
    "Abs_Coef": np.abs(log_clf.coef_[0])
}).sort_values("Abs_Coef", ascending=False)

print("\nTop 5 características por magnitud de coeficiente:")
print(coef_df.head(5)[["Característica", "Coeficiente"]])


Hiperparámetros de Regresión Logística:
{'class_weight': 'balanced', 'max_iter': 1000, 'random_state': 42, 'solver': 'lbfgs'}


Entrenamiento completado en 4.38 segundos.
Intercepto (bias): -0.7053
Número de coeficientes ajustados: 21

Top 5 características por magnitud de coeficiente:
                           Característica  Coeficiente
12   cat__mcc_category_Financial_Services    -4.352361
20    cat__mcc_category_Utilities_Telecom    -2.740477
10           cat__mcc_category_Car_Rental     2.554654
16  cat__mcc_category_Professional_Health    -2.181547
8      cat__mcc_category_Apparel_Clothing     2.143844


## 3. Entrenamiento del Modelo Comparativo: Regresión Lineal
Ajustamos una `LinearRegression` convencional por mínimos cuadrados ordinarios ($OLS$) sobre la variable objetivo binaria $\{0, 1\}$.
Este modelo se entrena con fines didácticos para comparar empíricamente sus deficiencias en evaluación.


In [5]:
print("Hiperparámetros de Regresión Lineal:")
print(LINEAR_REGRESSION_PARAMS)

lin_reg = LinearRegression(**LINEAR_REGRESSION_PARAMS)

t0 = time.time()
lin_reg.fit(X_train_trans, y_train)
t_train_lin = time.time() - t0

print(f"Entrenamiento de Regresión Lineal completado en {t_train_lin:.2f} segundos.")
print(f"Intercepto: {lin_reg.intercept_:.6f}")


Hiperparámetros de Regresión Lineal:
{'fit_intercept': True}


Entrenamiento de Regresión Lineal completado en 0.29 segundos.
Intercepto: 0.014464


## 4. Serialización de los Modelos Entrenados
Guardamos los objetos modelo en `artifacts/` para su posterior evaluación e inferencia.


In [6]:
save_artifact(log_clf, LOGISTIC_MODEL_ARTIFACT)
save_artifact(lin_reg, LINEAR_MODEL_ARTIFACT)

print(f"Modelo baseline (Regresión Logística) guardado en: {LOGISTIC_MODEL_ARTIFACT}")
print(f"Modelo comparativo (Regresión Lineal) guardado en: {LINEAR_MODEL_ARTIFACT}")


Modelo baseline (Regresión Logística) guardado en: /home/glender/ProyectosUNi/iaupeu/artifacts/baseline_logistic_regression.joblib
Modelo comparativo (Regresión Lineal) guardado en: /home/glender/ProyectosUNi/iaupeu/artifacts/linear_regression_comparison.joblib
